# Evaluation Notebook

In [2]:
import json
from ultralytics import YOLO
from pathlib import Path

DATA_ROOT = Path("Data/training_set").resolve() # main dataset root
COCO_JSON= DATA_ROOT / "slz_out" / "det_obb" / "slz_obb_all.json" # coco obb dataset json path

with open(COCO_JSON, "r") as f:
    coco = json.load(f)

# id -> name, then sorted by id to ensure correct order
id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
names = [id_to_name[i] for i in sorted(id_to_name)] # cls names


run_dir = Path("slz_obb/yolov11s_baseline2")   # <-- Model run dir
best = run_dir / "weights" / "best.pt"
print("Using best:", best, best.exists())


Using best: slz_obb\yolov11s_baseline2\weights\best.pt True


In [3]:
# Always use your new best weights
# adjust to your run folder
model = YOLO(best)

## yolo eval

In [ ]:
DATA_YAML = Path("Data/training_set/slz_out/det_obb/splits_yolo_obb/slz_obb_dataset.yaml").resolve() # Yolo config file

val_metrics  = model.val(data=str(DATA_YAML), split="val",  imgsz=1920, cache=True, project="slz_obb", name="val_metrics")
test_metrics = model.val(data=str(DATA_YAML), split="test", imgsz=1920, cache=True, project="slz_obb", name="test_metrics")
print("VAL mAP50-95:", val_metrics.box.map, "TEST mAP50-95:", test_metrics.box.map)


# Pretty-print some key numbers
def summarize(m):
    # m.box maps hold metrics for OBB task too
    print(f"mAP50-95: {m.box.map:.4f} | mAP50: {m.box.map50:.4f} | mAP75: {m.box.map75:.4f}")
    for i,cls_name in enumerate(names):
        print(f"  - {cls_name:>8s}: AP50-95={m.box.maps[i]:.4f}")

print("VAL metrics:");  summarize(val_metrics)
print("TEST metrics:"); summarize(test_metrics)

Ultralytics 8.3.226  Python-3.9.18 torch-2.0.1 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
YOLOv8n-obb summary (fused): 81 layers, 3,077,609 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access  (ping: 0.50.5 ms, read: 564.443.7 MB/s, size: 10182.3 KB)
val: Scanning E:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\Data\training_set\slz_out\det_obb\splits_yolo_obb\labels\val.cache... 40 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40  0.0s
WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.3GB RAM): 100% ━━━━━━━━━━━━ 40/40 19.1it/s 2.1s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 0.7it/s 4.4s2.2s
                   all         40        465       0.29       0.11      0.172     0.0936
                slz_l2         39        408     0.0793     

## Inference on test

In [ ]:
SPLIT_ROOT = Path("Data/training_set/slz_out/det_obb/splits_yolo_obb").resolve()  # Yolo-split root

pred_dir = SPLIT_ROOT / "images" / "test"
preds = model.predict(
    source=str(pred_dir),
    imgsz=1920,
    conf=0.25,
    iou=0.50,
    max_det=100,
    save=True,
    save_txt=True,
    project="slz_obb",
    name="11s_test_preds"
)
print("Predictions saved.")


image 1/40 E:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\038.jpg: 1280x1920 7 slz_l2s, 106.8ms
image 2/40 E:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\044.jpg: 1280x1920 10 slz_l2s, 29.6ms
image 3/40 E:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\049.jpg: 1280x1920 11 slz_l2s, 29.3ms
image 4/40 E:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\059.jpg: 1280x1920 3 slz_l2s, 30.0ms
image 5/40 E:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\063.jpg: 1280x1920 5 slz_l2s

## Selected Landing evaluation

In [1]:
from pathlib import Path
import numpy as np, cv2, math
from tqdm import tqdm
import os

# --------- CONFIG PATHS (from your description) ----------
DATA_ROOT   = Path("Data/training_set").resolve()
SPLIT_ROOT  = DATA_ROOT / "slz_out" / "det_obb" / "splits_yolo_obb"
TEST_IMG_DIR= SPLIT_ROOT / "images" / "test"
TEST_LBL_DIR= SPLIT_ROOT / "labels" / "test"
LEVELS_DIR  = DATA_ROOT / "slz_out" / "masks_levels"
PRED_ROOT   = Path("slz_obb/predictions").resolve()   # contains subdirs per model

# --------- METRIC CONSTANTS ----------
PURITY_MIN_SLS = 0.90   # SLS@L2 threshold
FSR_HAZARD_FRAC_MIN = 0.03  # 3% hazard-overlap to flag false-safe

def find_test_image(stem: str):
    # prefer .jpg, else any supported ext in TEST_IMG_DIR
    p = TEST_IMG_DIR / f"{stem}.jpg"
    if p.exists(): return p
    return None

def read_levels(stem: str) -> np.ndarray:
    p = LEVELS_DIR / f"{stem}_levels.png"
    lev = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if lev is None:
        raise FileNotFoundError(f"Missing levels mask: {p}")
    if lev.ndim == 3:  # BGR(A) -> R channel as used in your exporter
        lev = lev[:,:,2]
    return lev.astype(np.uint8)

def order_quad_clockwise(pts: np.ndarray) -> np.ndarray:
    # pts: (4,2) float, normalize ordering around centroid
    c = pts.mean(axis=0)
    ang = np.arctan2(pts[:,1]-c[1], pts[:,0]-c[0])
    order = np.argsort(ang)
    return pts[order]

def rasterize_quad_norm(pts_norm: np.ndarray, W: int, H: int) -> np.ndarray:
    # pts_norm in [0,1], shape (4,2)
    pts = pts_norm.copy()
    pts[:,0] = np.clip(pts[:,0] * W, 0, W-1)
    pts[:,1] = np.clip(pts[:,1] * H, 0, H-1)
    pts = order_quad_clockwise(pts).astype(np.float32)
    poly = np.zeros((H, W), np.uint8)
    cv2.fillConvexPoly(poly, pts.astype(np.int32), 1)
    return poly

def parse_pred_file(txt_path: Path):
    """Return list of dicts with keys: cls, conf, pts_norm(4,2)."""
    out = []
    if not txt_path.exists():
        return out
    for line in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s:
            continue
        parts = [float(x) for x in s.split()]
        if len(parts) < 9:
            continue  # not a valid OBB line
        cls = int(parts[0])
        if len(parts) == 9:
            conf = 1.0
            coords = parts[1:]
        else:
            conf = float(parts[1])
            coords = parts[2:]
        if len(coords) != 8:
            continue
        pts = np.array(coords, dtype=np.float32).reshape(4,2)
        out.append({"cls": cls, "conf": conf, "pts_norm": pts})
    return out

def box_area_from_mask(m: np.ndarray) -> int:
    return int(m.sum())

# --------- COLLECT TEST IDS ----------
test_ids = sorted([p.stem for p in TEST_LBL_DIR.glob("*.txt")])
assert test_ids, f"No test label files found in {TEST_LBL_DIR}"

# --------- PRELOAD GT (dims + masks) ----------
gt_cache = {}  # stem -> dict(H,W, levels_resized, safe2_mask, hazard0_mask, safe2_area)
for stem in tqdm(test_ids, desc="Preloading GT masks"):
    img_path = find_test_image(stem)
    if img_path is None:
        raise FileNotFoundError(f"No test image found for {stem} in {TEST_IMG_DIR}")
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"Cannot read test image {img_path}")
    H, W = img.shape[:2]
    levels = read_levels(stem)
    if levels.shape[:2] != (H, W):
        levels = cv2.resize(levels, (W, H), interpolation=cv2.INTER_NEAREST)
    safe2 = (levels >= 2).astype(np.uint8)
    hazard0 = (levels == 0).astype(np.uint8)
    gt_cache[stem] = dict(H=H, W=W, levels=levels, safe2=safe2, hazard0=hazard0, safe2_area=int(safe2.sum()))

# --------- ENUMERATE MODEL PREDICTION FOLDERS ----------
model_dirs = [d for d in PRED_ROOT.iterdir() if d.is_dir()]
assert model_dirs, f"No model folders found under {PRED_ROOT}"

summaries = []

for mdir in model_dirs:
    model_name = mdir.name
    print(f"\n=== Evaluating model: {model_name} ===")
    # per-model accumulators
    total_images = len(test_ids)
    images_with_preds = 0
    total_boxes = 0

    # mIoA (micro over all boxes)
    purity_sum_micro = 0.0

    # mIoA (macro over images with at least 1 box)
    purity_sum_macro = 0.0
    macro_count_images = 0

    # SAU (micro): numerator/denominator aggregated across images
    sau_safe_cover_sum = 0
    sau_safe_total_sum = 0

    # SLS + FSR
    sls_success = 0                  # denominator: total_images
    fsr_fail = 0                     # denominator: images_with_preds
    selected_count = 0               # images where we selected a box
    selected_purity_sum = 0.0        # for diagnostics
    selected_hazard_frac_sum = 0.0   # for diagnostics

    # iterate images
    for stem in tqdm(test_ids, desc=f"{model_name}", leave=False):
        gt = gt_cache[stem]
        H, W = gt["H"], gt["W"]
        levels = gt["levels"]
        safe2 = gt["safe2"]
        hazard0 = gt["hazard0"]
        safe2_area = gt["safe2_area"]

        # read predictions for this image
        pred_path = mdir / f"{stem}.txt"
        dets = parse_pred_file(pred_path)

        # --- Coverage metrics ---
        if dets:
            images_with_preds += 1

            # union for SAU
            union_mask = np.zeros((H, W), np.uint8)

            # per-image purity average (macro)
            purities_this_image = []

            for det in dets:
                cls = det["cls"]
                if cls not in (0,1):
                    continue  # ignore unexpected classes
                k = 2 if cls == 0 else 3

                P = rasterize_quad_norm(det["pts_norm"], W, H)
                A = box_area_from_mask(P)
                if A <= 0:
                    continue

                # purity p = |P ∩ (S >= k)| / |P|
                allowed = (levels >= k).astype(np.uint8)
                p = float((P & allowed).sum()) / float(A)
                purity_sum_micro += p
                purities_this_image.append(p)

                # union for SAU (always against S>=2)
                union_mask |= P

                total_boxes += 1

            # SAU numerator: |(∪P) ∩ (S>=2)|
            if safe2_area > 0:
                sau_safe_cover_sum += int((union_mask & safe2).sum())
                sau_safe_total_sum += safe2_area

            # per-image purity (macro)
            if purities_this_image:
                purity_sum_macro += float(np.mean(purities_this_image))
                macro_count_images += 1

        else:
            # no predictions: SAU denominator still counts safe area
            if safe2_area > 0:
                sau_safe_total_sum += safe2_area

        # --- Decision metrics (selection) ---
        # rank by Level-3 > Level-2, tie by area; if still tie, higher conf
        selected = None
        best_key = None
        for det in dets:
            cls = det["cls"]
            if cls not in (0,1):
                continue
            P = rasterize_quad_norm(det["pts_norm"], W, H)
            A = box_area_from_mask(P)
            if A <= 0:
                continue
            level_rank = 1 if cls == 1 else 0  # L3=1, L2=0
            key = (level_rank, A, det["conf"])
            if (best_key is None) or (key > best_key):
                best_key = key
                selected = (P, cls, A, det["conf"])

        if selected is None:
            # no selection => SLS failure for this image
            continue

        selected_count += 1
        Psel, cls_sel, A_sel, conf_sel = selected

        # Purity for SLS is vs S>=2 (operationally safe), not class-dependent
        p_sel = float((Psel & safe2).sum()) / float(A_sel)
        selected_purity_sum += p_sel

        # Hazard fraction for FSR
        hfrac_sel = float((Psel & hazard0).sum()) / float(A_sel)
        selected_hazard_frac_sum += hfrac_sel

        if p_sel >= PURITY_MIN_SLS:
            sls_success += 1

        if hfrac_sel >= FSR_HAZARD_FRAC_MIN:
            fsr_fail += 1

    # ---- Aggregate per-model ----
    mIoA_micro = (purity_sum_micro / total_boxes) if total_boxes > 0 else 0.0
    mIoA_macro = (purity_sum_macro / macro_count_images) if macro_count_images > 0 else 0.0
    SAU = (sau_safe_cover_sum / sau_safe_total_sum) if sau_safe_total_sum > 0 else 0.0
    SLS_L2 = sls_success / total_images
    FSR = (fsr_fail / selected_count) if selected_count > 0 else 0.0
    det_rate = images_with_preds / total_images
    avg_sel_purity = (selected_purity_sum / selected_count) if selected_count > 0 else 0.0
    avg_sel_hfrac  = (selected_hazard_frac_sum / selected_count) if selected_count > 0 else 0.0

    summary = dict(
        model=model_name,
        images=total_images,
        imgs_with_preds=images_with_preds,
        det_rate=round(det_rate, 3),
        boxes=total_boxes,
        mIoA_micro=round(mIoA_micro, 4),
        mIoA_macro=round(mIoA_macro, 4),
        SAU=round(SAU, 4),
        SLS_L2=round(SLS_L2, 4),
        FSR=round(FSR, 4),
        avg_sel_purity=round(avg_sel_purity, 4),
        avg_sel_hfrac=round(avg_sel_hfrac, 4),
    )
    summaries.append(summary)

# ---- Print summary table ----
cols = ["model","images","imgs_with_preds","det_rate","boxes",
        "mIoA_micro","mIoA_macro","SAU","SLS_L2","FSR","avg_sel_purity","avg_sel_hfrac"]
hdr = " | ".join([f"{c:>15s}" for c in cols])
print("\n" + hdr)
print("-"*len(hdr))
for s in sorted(summaries, key=lambda d: d["mIoA_micro"], reverse=True):
    print(" | ".join([f"{str(s[c]):>15s}" for c in cols]))


Preloading GT masks: 100%|██████████| 40/40 [00:17<00:00,  2.33it/s]



=== Evaluating model: 11n ===



=== Evaluating model: 11s ===



=== Evaluating model: 11s-2 ===



=== Evaluating model: 5s ===



=== Evaluating model: 8n ===



=== Evaluating model: 8n-2 ===



=== Evaluating model: 8n-6 ===



          model |          images | imgs_with_preds |        det_rate |           boxes |      mIoA_micro |      mIoA_macro |             SAU |          SLS_L2 |             FSR |  avg_sel_purity |   avg_sel_hfrac
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
           8n-6 |              40 |              31 |           0.775 |              68 |          0.8817 |          0.8785 |          0.2768 |             0.6 |             0.0 |          0.9159 |          0.0014
            11s |              40 |              16 |             0.4 |              23 |          0.8807 |          0.8572 |           0.238 |            0.25 |          0.0625 |          0.8658 |          0.0106
          11s-2 |              40 |              38 |            0.95 |             259 |          0.8663 |          0.8695 |          0.4588 |